# Case study: Dual-Level Retrieval vs. Baseline Hybrid RAG

Compares two retrieval architectures on ONE question, side by side:

- **Baseline** -- Hybrid RAG only (`run_hybrid_only_question.py`): BM25 + dense vector over the full Azure AI Search index, no knowledge graph involved.
- **Dual-Level Retrieval** -- LightRAG local (1-hop neighborhood) + global (semantic relation) retrieval, fused with Hybrid RAG (`run_dual_level_hybrid_question.py`) -- the same combination `orchestrator.py` uses for the `aggregation` retrieval-need route.

Scored on 5 custom dimensions (not ragas's built-in metrics -- purpose-built for this comparison, structured in the same spirit as ragas's pointwise LLM-judge prompts):

| Metric | What it evaluates | Why it matters |
|---|---|---|
| **Completeness** | Does the answer address all aspects of the question? | Multi-aspect engineering questions |
| **Retrieval Coverage** | Does retrieval gather all necessary supporting evidence? | Measures retrieval effectiveness |
| **Groundedness** | Are all claims supported by retrieved documents? | Prevents hallucination |
| **Relevance** | Is the answer focused on the requested engineering problem? | Avoids unnecessary information |
| **Cross-document Integration** | Does the answer successfully combine evidence from multiple documents? | Highlights the advantage of Dual-Level Retrieval |

Question under study (deliberately multi-aspect -- "height-based requirements" AND "spill control considerations" -- and plausibly cross-document, since ACC height requirements and spill containment are often specified in different exhibits):

> What are the height-based requirements and spill control considerations for the Air-Cooled Condenser (ACC) in the tender context?

The judge uses this app's existing APIM-authenticated LLM client (`src.pipeline.clients.call_json`), same as `_llm_judge.py` -- not a separate `OPENAI_API_KEY` setup, since this is a custom judge in that same family, not a ragas metric.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Dict, List

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from pydantic import BaseModel

from src.pipeline.clients import call_json, validate_model
from run_hybrid_only_question import run_hybrid_only
from run_dual_level_hybrid_question import run_dual_level_hybrid

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


/opt/anaconda3/envs/gds_test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QUESTION = (
    "Summarize all engineering, safety, and environmental considerations related to the Air-Cooled Condenser (ACC)."
)

## Judge: 5-dimension case-study scorer

One LLM call scores all 5 dimensions together (same pattern as `_llm_judge.py`'s 4-dimension judge -- cheaper and more consistent than 5 separate calls). Each dimension's instructions spell out exactly what distinguishes it from the others -- especially **completeness** (answer-level) vs. **retrieval_coverage** (context-level), which are easy to conflate but measure different pipeline stages.

In [3]:
class CaseStudyScore(BaseModel):
    completeness: int
    retrieval_coverage: int
    groundedness: int
    relevance: int
    cross_document_integration: int
    reasoning: str = ""


CASE_STUDY_JUDGE_SYS = (
    "You are an impartial evaluator comparing two retrieval-augmented generation (RAG) "
    "architectures for a tender/ITB (Invitation to Bid) engineering question-answering "
    "assistant: Hybrid RAG (a baseline that searches an Azure AI Search index with BM25 + "
    "dense vector retrieval) versus Dual-Level Retrieval (a knowledge-graph method that "
    "retrieves via LightRAG's local 1-hop neighborhood and global semantic relation channels, "
    "fused with Hybrid RAG). You score ONE answer to ONE question in isolation, using ONLY the "
    "retrieved contexts provided -- do not use outside knowledge of the true facts, and do not "
    "compare this answer to any other answer. Return strict JSON only."
)

CASE_STUDY_JUDGE_USER = """Question:
{question}

Retrieved contexts (the evidence available to the system when it generated the answer,
one per source file/chunk):
{contexts_block}

Distinct source files represented in the retrieved contexts above: {file_list}

Answer:
{answer}

Score the answer on each dimension below, 1-5 (5 = best). Base every score ONLY on the
question, the retrieved contexts, and the answer given -- not on outside/expert knowledge
of the true facts, since your job is to judge THIS pipeline's behavior, not fact-check it
against ground truth you don't have.

- completeness (1-5): Does the ANSWER address EVERY distinct aspect the question asks
  about? Multi-aspect questions (e.g. "X and Y considerations") require the answer to
  cover ALL named aspects, not just the first or easiest one. 5 = every aspect is
  addressed; 1 = the answer only addresses a single aspect while ignoring others the
  question explicitly asked about.

- retrieval_coverage (1-5): Do the RETRIEVED CONTEXTS ABOVE (not the answer) contain
  enough evidence to fully address every aspect of the question? This scores the
  RETRIEVAL step, not the generation step -- score this LOW if the contexts are missing
  evidence for one of the question's aspects, even if the answer sounds confident; score
  this HIGH if the contexts genuinely contain what's needed, regardless of whether the
  answer actually used it well (that's what completeness measures instead).

- groundedness (1-5): Is every factual claim in the answer directly supported by the
  retrieved contexts? 5 = no claim goes beyond what the contexts state; 1 = the answer
  contains claims that are fabricated, unsupported, or contradicted by the contexts. Do
  not penalize reasonable synthesis or paraphrasing of what the contexts actually say --
  only penalize claims the contexts do not support at all.

- relevance (1-5): Is the answer tightly focused on the engineering problem the question
  actually asks, without padding, tangents, or unrequested information?

- cross_document_integration (1-5): Does the answer meaningfully COMBINE evidence from
  MULTIPLE DISTINCT SOURCE FILES -- not just cite several files trivially, but actually
  synthesize facts that individually come from different documents into one coherent
  answer? 1 = the answer only draws on a single file, or multiple files were available but
  the answer ignores that and only uses one; 5 = the answer genuinely integrates
  information across two or more of the distinct files listed above into a unified answer,
  where that cross-document synthesis is essential (not incidental) to answering the
  question. If only one distinct file is listed above, this dimension cannot be
  demonstrated regardless of the answer -- score it 1 and say so in the reasoning, rather
  than penalizing or crediting the answer for something retrieval never made possible.

Return STRICT JSON only:
{{"completeness": <1-5>, "retrieval_coverage": <1-5>, "groundedness": <1-5>, "relevance": <1-5>, "cross_document_integration": <1-5>, "reasoning": "<2-4 sentences justifying each score, referencing specific evidence gaps or integration points>"}}
"""


def _contexts_block(contexts: List[Dict[str, Any]]) -> str:
    lines = []
    for i, c in enumerate(contexts, 1):
        fn = c.get("file_name", "Unknown")
        pn = c.get("page_number", "N/A")
        content = (c.get("content") or "").strip()[:800]
        lines.append(f"[{i}] {fn} (p.{pn})\n{content}")
    return "\n\n".join(lines) if lines else "(no contexts retrieved)"


def judge_case_study(question: str, contexts: List[Dict[str, Any]], answer: str) -> CaseStudyScore:
    file_list = sorted({c.get("file_name", "Unknown") for c in contexts})
    user_prompt = CASE_STUDY_JUDGE_USER.format(
        question=question,
        contexts_block=_contexts_block(contexts),
        file_list=file_list,
        answer=answer or "(no answer)",
    )
    data = call_json(CASE_STUDY_JUDGE_SYS, user_prompt)
    return validate_model(data, CaseStudyScore)

## Run both pipelines on the same question

In [5]:
print("Running Hybrid-RAG-only baseline...")
baseline_result = run_hybrid_only(QUESTION)
baseline_answer = baseline_result["answer"]
baseline_contexts = baseline_result["reranked_chunks"]
print(f"  {len(baseline_contexts)} context chunks, {len(set(c.get('file_name') for c in baseline_contexts))} distinct files")
print(f"  Answer: {baseline_answer.answer[:300]}")

Running Hybrid-RAG-only baseline...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 18828.73it/s]


  20 context chunks, 10 distinct files
  Answer: ### 📝 Engineering Considerations
- **Material Selection**: Use of 316L stainless steel for high-temperature applications (above 50°C–60°C) to manage corrosion and ensure durability [Source 1, Source 2].
- **Thermal and Acoustic Insulation**: Insulation systems must retain thermal energy, manage acou


In [6]:
print(baseline_answer.answer)

### 📝 Engineering Considerations
- **Material Selection**: Use of 316L stainless steel for high-temperature applications (above 50°C–60°C) to manage corrosion and ensure durability [Source 1, Source 2].
- **Thermal and Acoustic Insulation**: Insulation systems must retain thermal energy, manage acoustic hazards, and exclude environmental moisture to prevent corrosion [Source 3].
- **Hot Air Recirculation (HAR) Study**: Assess the impact of hot air recirculation on plant operations, including the interface between AGRU and CCS facilities [Source 4, Source 5].
- **Footprint Optimization**: Evaluate air-cooled heat exchanger (ACHE) technology to reduce module size and optimize hydraulics balance [Source 6].
- **HVAC Design**: Ensure proper HVAC design, including equipment location and integration with air-cooled condensers [Source 4, Source 5].

### ✅ Safety Considerations
- **Spill Control**: Design spill control systems to prevent cooling medium leaks from contaminating non-contaminated

In [ ]:
12, 15,123,68,129,134,8,18

In [15]:
baseline_contexts

[{'chunk_id': 'ROC_INPEX_Inpex_AGRU_CCS__FEED_validation_contract_Received_via_Ariba_02_Minor_Contract_Exhibit-A_Scope_of_Work_pdf_size_407543_page_12_text_chunk_2',
  'content': 'exceedance events) \n➢ Materials Design \n• \nProvides material selection that supports the piping and equipment design \nto achieve the Process Design intent for all foreseen operational scenarios \n• \nProvides material selection or other design considerations to manage \ncorrosion throughout the facility design life \n• \nProvides insulation system selection and application to meet the intents of \nthermal energy retention or exclusion, management of acoustic hazards to \nmeet regional regulatory obligations, and exclusion of environmental \nmoisture to support corrosion management  \n➢ Technical Safety Design \n• \nDesign supports alarming in the event of hazardous situations and enables \nassessment of same by Plant Operations/emergency response personnel, \nnotification of hazardous situations to affect

In [4]:
print("Running Dual-Level Retrieval + Hybrid RAG...")
dual_result = run_dual_level_hybrid(QUESTION)
dual_answer = dual_result["answer"]
dual_contexts = dual_result["reranked_chunks"]
print(f"  {len(dual_contexts)} context chunks, {len(set(c.get('file_name') for c in dual_contexts))} distinct files")
print(f"  Answer: {dual_answer.answer[:300]}")

Running Dual-Level Retrieval + Hybrid RAG...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n            CALL db.index.vector.queryNodes('relation_embedding_index', $query_k, $qvec)\n            YIELD node AS r, score\n            WHERE ($domains IS NULL OR size([(r)-[:CLASSIFIED_AS]->(dom) WHERE dom.domainName IN $domains | dom]) > 0)\n            MATCH (s:Entity)-[:SUBJECT_OF]->(r)-[:OBJECT_OF]->(o:Entity)\n            OPTIO

  50 context chunks, 19 distinct files
  Answer: ### 📝 Engineering Considerations
- **Design Optimization**: Evaluate air-cooled heat exchanger (ACHE) technology to reduce footprint requirements, improve fan and finned tube designs, and ensure hydraulic balance between existing and new cooling systems [Source 1].
- **Dynamic Studies**: Perform Hot


In [ ]:
12, 15,123,68,129,134,8,18

In [6]:
dual_contexts

[{'chunk_id': 'ROC_INPEX_2_0_Tender_Supplements_WS33071728_-TB-0011_Tender_Supplement__5_-_Updated_Design_Dossier__Aug25__pdf_size_1952094_page_110_text_chunk_2',
  'content': '24/10/2024\nN/A\n3_ITT\n16/12/2024\nE10-01-006\nTQD-999996-H3-00025\nLocation of L552-TP-523A/B and L552-TP-526A/B \nand assoc. relocation of escape route\n1\nHealth, Safety, \nEnvironment and \nSecurity\nTechnical Safety\nReturned\n3, TQ Classification Only \n(Package Approval)\nAC, Accepted\nNOT CLOSED\nCompany\n3_ITT\n16/12/2024\nE10-01-007\nTQD-999996-H3-00026\nLocation of AGRU Semi-Lean Knock-Out Pot (552-V-\n010) and AGRU Semi-Lean Drain Drum (552- V-011) \nunder air-fin coolers\n0\nHealth, Safety, \nEnvironment and \nSecurity\nTechnical Safety\nReturned\n2A, TD Approved\nACC, Accepted \nWith Comments\nNOT CLOSED\nEPC\nImplement proposal as per TQD\n3_ITT\n16/12/2024\nE10-01-008\nTQD-999996-X1-00027\nExpansion Joints for CCS IG Compressor Nozzle\n0\nPiping Engineering\nPiping Engineering\nReturned\n2A, TD 

In [7]:
print(dual_answer.answer)

### 📝 Engineering Considerations
- **Design Optimization**: Evaluate air-cooled heat exchanger (ACHE) technology to reduce footprint requirements, improve fan and finned tube designs, and ensure hydraulic balance between existing and new cooling systems [Source 1].
- **Dynamic Studies**: Perform Hot Air Recirculation (HAR) studies to assess the impact of ACC placement on existing facilities and optimize module height to minimize structural steel usage [Source 2, Source 1].
- **Material Selection**: Ensure corrosion control and insulation systems meet thermal energy retention, acoustic hazard management, and environmental moisture exclusion requirements [Source 3].
- **Process Safety**: Conduct simulations for loss of containment scenarios, tube rupture analysis, and vibration assessments to avoid fatigue failure in process pipework [Source 2, Source 4].

### ✅ Safety Considerations
- **Hazard Identification**: Perform HAZID and HAZOP studies to identify and mitigate risks associated wi

## Score both conditions

In [8]:
baseline_score = judge_case_study(QUESTION, baseline_contexts, baseline_answer.answer)
dual_score = judge_case_study(QUESTION, dual_contexts, dual_answer.answer)

print("Baseline (Hybrid RAG only):", baseline_score.model_dump())
print()
print("Dual-Level Retrieval + Hybrid:", dual_score.model_dump())

Baseline (Hybrid RAG only): {'completeness': 3, 'retrieval_coverage': 3, 'groundedness': 2, 'relevance': 4, 'cross_document_integration': 3, 'reasoning': "The answer addresses engineering, safety, and environmental considerations but lacks depth in some areas, such as specific environmental impacts and detailed safety measures. The retrieved contexts provide partial evidence for the question but are missing key details, such as comprehensive environmental considerations and specific ACC-related safety measures. Some claims in the answer, such as 'Hot Air Recirculation (HAR) Study' and 'Energy Efficiency,' are not directly supported by the retrieved contexts, reducing groundedness. The answer is relevant to the question, focusing on the requested aspects without unnecessary tangents. It integrates information from multiple sources, but the synthesis is limited and does not fully leverage all available evidence."}

Dual-Level Retrieval + Hybrid: {'completeness': 4, 'retrieval_coverage': 

## Side-by-side comparison

In [9]:
dimensions = ["completeness", "retrieval_coverage", "groundedness", "relevance", "cross_document_integration"]

print("=" * 100)
print(f"QUESTION: {QUESTION}")
print("=" * 100)
print()
print(f"{'Dimension':<30}{'Baseline (Hybrid)':<22}{'Dual-Level + Hybrid':<22}{'Delta'}")
print("-" * 100)
for dim in dimensions:
    b, d = getattr(baseline_score, dim), getattr(dual_score, dim)
    print(f"{dim:<30}{b:<22}{d:<22}{d - b:+d}")

print()
print("-" * 100)
print("BASELINE (Hybrid RAG only) reasoning:")
print(f"  {baseline_score.reasoning}")
print(f"  Distinct files used: {sorted(set(c.get('file_name') for c in baseline_contexts))}")
print()
print("DUAL-LEVEL + HYBRID reasoning:")
print(f"  {dual_score.reasoning}")
print(f"  Distinct files used: {sorted(set(c.get('file_name') for c in dual_contexts))}")
print("=" * 100)

print()
print("BASELINE ANSWER:")
print(baseline_answer.answer)
print()
print("DUAL-LEVEL ANSWER:")
print(dual_answer.answer)

QUESTION: Summarize all engineering, safety, and environmental considerations related to the Air-Cooled Condenser (ACC).

Dimension                     Baseline (Hybrid)     Dual-Level + Hybrid   Delta
----------------------------------------------------------------------------------------------------
completeness                  3                     4                     +1
retrieval_coverage            3                     4                     +1
groundedness                  2                     3                     +1
relevance                     4                     5                     +1
cross_document_integration    3                     4                     +1

----------------------------------------------------------------------------------------------------
BASELINE (Hybrid RAG only) reasoning:
  The answer addresses engineering, safety, and environmental considerations but lacks depth in some areas, such as specific environmental impacts and detailed safety measu

In [ ]:
RESULTS_PATH = CHATBOT_ROOT / "domain_data" / "eval_results" / "case_study_dual_level_vs_hybrid.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

report = {
    "question": QUESTION,
    "baseline": {
        "answer": baseline_answer.answer,
        "n_contexts": len(baseline_contexts),
        "distinct_files": sorted(set(c.get("file_name") for c in baseline_contexts)),
        "score": baseline_score.model_dump(),
    },
    "dual_level": {
        "answer": dual_answer.answer,
        "n_contexts": len(dual_contexts),
        "distinct_files": sorted(set(c.get("file_name") for c in dual_contexts)),
        "score": dual_score.model_dump(),
    },
}
RESULTS_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote case study report -> {RESULTS_PATH}")